# Proyecto de Analítica y Modelado de Datos: Supermercado
## Etapa 3: Creación de tablas

### Alcance y Objetivos

## Alcance del trabajo de creación de tablas

El objetivo fue construir las seis tablas del diagrama relacional (`producto`, `cliente`, `venta`, `detalle_venta`, `cuenta_por_cobrar` y `gastos_insumo`) a partir del archivo real `ventas_originales_fuente.csv` (103.489 líneas de venta), completando con datos ficticios lo que la fuente no contiene.

**Carga y preparación:** el archivo se leyó sin modificar (`df_origen`, como respaldo), leyendo `_id`, `factura` y `no_parte` como texto, y se trabajó sobre una copia (`df`) donde la fecha se convirtió de texto ISO a fecha, sin zona horaria ni hora.

**Revisión de calidad:** se verificó que no hubiera fechas inválidas, se revisaron registros, unidades e ingresos por año y por mes, y se detectó que los datos posteriores al 31 de diciembre de 2020 tienen una cobertura parcial. Se marcaron en una columna auxiliar `cobertura` (completa / parcial) del DataFrame de trabajo, pero se conservaron en todas las tablas, por lo que el rango final abarca de 2010 a 2022.

**Producto:** se comprobó que cada `no_parte` tuviera un único nombre y una única unidad de negocio, y que los nombres repetidos con códigos distintos no afectaran la identificación; por eso el número de parte se usa directamente como `id_producto`. La `categoria` se asignó explorando primero las palabras más frecuentes de los nombres y luego clasificando con reglas de palabras clave: el texto se normaliza (minúsculas, sin tildes) y se aplican reglas por niveles de prioridad, ganando la primera que coincide. Se definieron 13 categorías (Abarrotes, Aseo del hogar, Cuidado personal, Snacks y confitería, entre otras); lo que no coincide queda como "Otros / sin clasificar", y el resultado se revisó con el porcentaje de productos clasificados, los ingresos cubiertos y una muestra de control por categoría. La categoría queda como columna dentro de `producto`, sin tabla aparte.

**Venta y detalle:** provienen de las ventas reales (cantidad, precio, costo, fecha y tipo de compra); `id_factura` se construyó con prefijo, fecha y tipo de compra, porque el prefijo solo no identifica una factura única, y resultaron 100.067 facturas.

**Datos ficticios:** `cliente` (2.000 clientes con nombre y teléfono), la asignación de cliente a cada factura, `por_cobrar` (15 %), `fecha_cobro` (15 a 60 días después de la venta) y `gastos_insumo` (20.000 registros con montos basados en el costo promedio del producto).

**Cuenta por cobrar:** se deriva de las facturas por cobrar (14.928), y su `monto` se calcula con los datos reales de `detalle_venta` (cantidad × precio de venta).

**Validación y ajustes:** todas las tablas se validaron en el código (claves únicas, claves foráneas existentes, sin nulos, valores positivos y largos dentro de los límites del diagrama) y se exportaron como CSV. Como ajuste al diagrama, `id_producto` pasa a texto, y `id_detalle`, `id_cuenta` e `id_gasto` necesitan más dígitos de los previstos.

In [ ]:
import pandas as pd
import numpy as np
import sqlite3
from pathlib import Path

pd.set_option("display.max_columns", None)
pd.set_option("display.float_format", "{:,.2f}".format)

# El archivo debe estar subido en la misma carpeta del notebook (panel Files)
ruta = Path("ventas_originales_fuente.csv")

if not ruta.exists():
    print("Archivos en la carpeta actual:", [p.name for p in Path(".").iterdir()])
    raise FileNotFoundError(f"No se encontró '{ruta}'. Súbelo desde el panel Files.")

# Lectura sin modificar: se conserva como respaldo del origen
df_origen = pd.read_csv(
    ruta,
    encoding="utf-8",
    dtype={"_id": str, "factura": str, "no_parte": str},
)

print(f"Archivo cargado: {ruta.name}")
print(f"Filas: {df_origen.shape[0]:,} | Columnas: {df_origen.shape[1]}")
df_origen.head()

Archivo cargado: ventas_originales_fuente.csv
Filas: 103,489 | Columnas: 12


,_id,fecha,factura,no_parte,producto,unidad_negocio,tipo_compra,cantidad,precio_venta,costo_venta,source_row_id,replica_id
0,697a14799bf21b7e4edd8b08,2020-06-16T00:00:00.000Z,INV7778844-227330,P373917,Alimento Lácteo Finesse,Supermercado,Física,1,665,332,0,227330
1,697a14619bf21b7e4edbfcff,2020-04-02T00:00:00.000Z,INV7779415-125433,P373917,Alimento Lácteo Finesse,Supermercado,Física,1,665,332,1,125433
2,697a14889bf21b7e4ede7edd,2020-04-22T00:00:00.000Z,INV7783006-289751,P373917,Alimento Lácteo Finesse,Supermercado,Física,1,665,332,10,289751
3,697a14729bf21b7e4edd1d8d,2012-08-24T00:00:00.000Z,INV7805885-199303,P373917,Alimento Lácteo Finesse,Supermercado,Física,1,1080,556,100,199303
4,697a14479bf21b7e4eda4d1b,2019-04-04T00:00:00.000Z,INV7788365-14869,P705069,Yogurt Griego Pasco Con Fresa,Supermercado,Física,1,81,70,1000,14869


In [ ]:
df_origen[["precio_venta","costo_venta"]].describe()

,precio_venta,costo_venta
count,"103,489.00","103,489.00"
mean,"1,332.37","1,045.38"
std,"1,641.53","1,333.06"
min,34.00,21.00
25%,196.00,155.00
50%,808.00,605.00
75%,"1,748.00","1,375.00"
max,"17,993.00","15,269.00"


Validación de información, de tipo cada columna y rango de fechas

In [ ]:
# Copia de trabajo (df_origen queda intacto como respaldo)
df = df_origen.copy()

# Fecha: de texto ISO a fecha, sin zona horaria ni hora
df["fecha"] = pd.to_datetime(df["fecha"], errors="coerce", utc=True).dt.tz_localize(None).dt.normalize()

# Verificación
print("Fechas no válidas:", df["fecha"].isna().sum())
print("Rango de fechas:", df["fecha"].min().date(), "a", df["fecha"].max().date())
print(df.dtypes)
print(df["tipo_compra"].unique())
print(df["unidad_negocio"].unique())

Fechas no válidas: 0
Rango de fechas: 2010-01-05 a 2022-11-18
_id                       object
fecha             datetime64[ns]
factura                   object
no_parte                  object
producto                  object
unidad_negocio            object
tipo_compra               object
cantidad                   int64
precio_venta               int64
costo_venta                int64
source_row_id              int64
replica_id                 int64
dtype: object
['Física' 'En Línea']
['Supermercado']


Registros: el archivo tiene 103.489 filas, se realizará el análisis de duplicados.

In [ ]:
# Registros y ventas por año
resumen_anual = (
    df.assign(anio=df["fecha"].dt.year)
      .groupby("anio")
      .agg(registros=("factura", "count"),
           unidades=("cantidad", "sum"),
           ingreso=("precio_venta", lambda s: (s * df.loc[s.index, "cantidad"]).sum()))
)
print(resumen_anual)

# Meses con datos por año (detecta años incompletos)
print(df.groupby(df["fecha"].dt.year)["fecha"].apply(lambda s: s.dt.month.nunique()))

      registros  unidades   ingreso
anio                               
2010       3594     10948  16398987
2011       5732     20163  28843919
2012       6544     23683  34983230
2013       6354     21838  30021338
2014       6446     23671  31399284
2015       9600     34808  37590937
2016      10409     36300  41162356
2017       9793     32070  33911203
2018      13979     51839  49800798
2019      17068     58968  57385826
2020      11533     35843  35419693
2021       2196      6788   6066628
2022        241       789    536824
fecha
2010    12
2011    12
2012    12
2013    12
2014    12
2015    12
2016    12
2017    12
2018    12
2019    12
2020    12
2021    12
2022    11
Name: fecha, dtype: int64


Los totales cuadran con la carga: la suma de registros da 103.489. Sin embargo, 2021 y 2022 tienen datos en todos los meses con muy pocos registros por mes, lo que sugiere un corte o muestreo de la fuente más que una caída real del negocio. Se debe verificar volumen por mes.

In [ ]:
# Dónde empieza la caida?
# Registros por mes, 2019 a 2022
mensual = (
    df[df["fecha"].dt.year >= 2019]
    .groupby([df["fecha"].dt.year.rename("anio"), df["fecha"].dt.month.rename("mes")])
    .size()
    .unstack("mes", fill_value=0)
)
print(mensual)

mes     1     2     3     4     5     6     7     8     9     10    11    12
anio                                                                        
2019  1470  1205  1492  1570  1610  1477  1517  1501  1383  1277  1189  1377
2020  1159   971  1094   951   957   898   972   904   972   968   905   782
2021   529   259   220   225   201   189   164   152    94    65    50    48
2022    43    31    31    27    27    28    19    19     9     6     1     0


Se conserva todos los registros en las tablas y se marca 2021-2022 como cobertura parcial y se excluye de tendencias, flujo de caja y comparaciones anuales. 2020 se mantiene, porque su nivel es bajo respecto de 2019 pero estable durante los 12 meses.

In [ ]:
FECHA_CORTE = pd.Timestamp("2020-12-31")

df["cobertura"] = np.where(df["fecha"] <= FECHA_CORTE, "completa", "parcial")

print(df["cobertura"].value_counts())
print(df.groupby("cobertura")["fecha"].agg(["min", "max"]))

cobertura
completa    101052
parcial       2437
Name: count, dtype: int64
                 min        max
cobertura                      
completa  2010-01-05 2020-12-31
parcial   2021-01-01 2022-11-18


In [ ]:
# 1. Verificar que cada no_parte tenga un único nombre y una única unidad de negocio
chequeo = df.groupby("no_parte").agg(
    nombres=("producto", "nunique"),
    unidades=("unidad_negocio", "nunique"),
    registros=("factura", "count"),
)
print("Productos distintos (no_parte):", len(chequeo))
print("no_parte con más de un nombre:", (chequeo["nombres"] > 1).sum())
print("no_parte con más de una unidad de negocio:", (chequeo["unidades"] > 1).sum())

# 2. Verificar el caso inverso: un mismo nombre con códigos distintos
nombres_repetidos = df.groupby("producto")["no_parte"].nunique()
print("Nombres con más de un no_parte:", (nombres_repetidos > 1).sum())

Productos distintos (no_parte): 7488
no_parte con más de un nombre: 0
no_parte con más de una unidad de negocio: 0
Nombres con más de un no_parte: 1029


In [ ]:
# 3. Construir la tabla (si un no_parte tuviera varios nombres, se usa el más frecuente)
producto = (
    df.groupby("no_parte")
      .agg(producto=("producto", lambda s: s.mode().iloc[0]),
           unidad_negocio=("unidad_negocio", lambda s: s.mode().iloc[0]))
      .reset_index()
)

# La categoría se asignará al crear la tabla categoria (siguiente paso)
producto["id_categoria"] = pd.NA

# 4. Validaciones de integridad
assert producto["no_parte"].is_unique, "no_parte duplicado"
assert producto["no_parte"].notna().all(), "no_parte con nulos"
assert set(df["no_parte"]) == set(producto["no_parte"]), "Hay productos sin cargar"

print(f"Tabla producto creada: {producto.shape[0]:,} filas")
producto.head()

Tabla producto creada: 7,488 filas


,no_parte,producto,unidad_negocio,id_categoria
0,P100055,Pasta Capelli D´Angelo Conzazoni,Supermercado,<NA>
1,P100083,Salchicha Rica,Supermercado,<NA>
2,P100512,Maizitos,Supermercado,<NA>
3,P100675,Leche de coco Badia,Supermercado,<NA>
4,P100765,Guarda polvo microfibra Ubl,Supermercado,<NA>


Tabla producto creada y validada: 7.488 filas, tres validaciones de integridad sin errores. Coincide con los 7.488 no_parte distintos del chequeo, así que no se perdió ni duplicó ningún producto. id_categoria aparece vacío (<NA>), como corresponde por ahora.

Falta el resultado de la celda 5, el diagnóstico de los 1.029 nombres repetidos con códigos distintos. No bloquea la tabla, pero define cómo asignaremos las categorías: si son variantes legítimas, se clasifican por nombre y se propaga a todos sus códigos.
Con eso pasamos a la tabla categoria. Clasificar 7.488 productos a mano no es viable, así que propondré un enfoque por palabras clave sobre el nombre (por ejemplo “leche”, “yogurt” o “champú”), con una categoría “Sin clasificar” para lo que no calce y una revisión de los casos más frecuentes.

In [ ]:
import re
from collections import Counter

stop = {"de","la","el","los","las","con","para","en","y","del","por","sin",
        "una","gr","ml","kg","lt","pack","pzas","pza","oz"}

def palabras(texto):
    return [w for w in re.findall(r"[a-záéíóúñü]{3,}", texto.lower()) if w not in stop]

# Palabras más frecuentes: en cuántos productos aparece cada una
frec = Counter(w for nombre in producto["producto"] for w in set(palabras(nombre)))
top = pd.DataFrame(frec.most_common(100), columns=["palabra", "productos"])
print(top.to_string(index=False))

# Muestra aleatoria de nombres
print()
print(producto["producto"].sample(40, random_state=1).to_string(index=False))

         palabra  productos
           vogue        317
         esmalte        307
            vino        305
           crema        300
           queso        286
       chocolate        231
             und        202
           bolsa        190
          yogurt        174
           leche        172
          bebida        159
         botella        157
       fantastic        153
             gel        145
         natural        135
           pasta        132
          azúcar        128
         liquido        127
           limón        127
          alpina        126
           fresa        122
           polvo        120
           jabón        118
        vainilla        117
           salsa        117
             pan        117
        repuesto        113
         galleta        111
        galletas        110
           barra        108
          blanco        108
         cerveza        105
     desodorante        102
         líquido        102
         surtido    

Ahora:  Definir las categorías de ingreso (por ejemplo lácteos, abarrotes, limpieza o cuidado personal) a partir de las palabras más frecuentes.
Armar un diccionario de palabras clave por categoría y asignar la categoría a cada producto por coincidencia en el nombre.
Medir la cobertura: los productos sin coincidencia quedan como “Sin clasificar”, y se revisa los más vendidos para ampliar las reglas hasta cubrir la mayor parte de las ventas.
Crear la tabla categoria con su tipo = ingreso, y completar id_categoria en producto.

In [ ]:
import re, unicodedata

def normalizar(t):
    t = unicodedata.normalize("NFKD", str(t).lower().replace("ÿ", " "))
    return "".join(c for c in t if not unicodedata.combining(c))

# ── 1. Reglas, de mayor a menor prioridad: gana la primera que coincide ─────────────
PRIORIDAD = [   # específicas: ganan sobre palabras genéricas
             ("Snacks y confitería",  ["palitos de queso"]),
 ("Cuidado personal",     ["carey","jabon nutritivo","leche corporal","babaria"]),
 ("Licores y cervezas",   ["tinto verano","carinoso"]),
 ("Bebidas sin alcohol",  ["jugo"]),
 ("Belleza y maquillaje", ["caiman","caimanes","caucho"]),
 ("Aseo del hogar",       ["mr musculo","sanitario","discos activos"]),
 ("Snacks y confitería",  ["crokante","papas","papa natural","natuchips","hojuelas de platano","caladitos",
                           "palitroques","maranon","macadamia","bocadillo","menta chao"]),
 ("Bebidas sin alcohol",  ["te","aromatica","infusion","hindu","jaibol","fuze","finest call","granadina",
                           "jarabe","gatorade","powerade","zumo","nectar","hit lulo"]),
 ("Abarrotes",            ["nescafe","cafe","sello rojo","sopera","gnochi","gnocchi","croqueta","papa francesa",
                           "naturavena","cuchuco","endulzante","pate","champinones","lasagna","curry","anis",
                           "sazonarey","rey bolsa","finas hierbas"]),
 ("Panadería y galletas", ["brioche","ring cake","ponky","ponke","pantegral"]),
 ("Licores y cervezas",   ["aperitivo","sangria"]),
 ("Lácteos y huevos",     ["quesillo"]),
 ("Cuidado personal",     ["toallas higienicas","tena"]),
 ("Belleza y maquillaje", ["ruedita","pomys","desmaquilladoras"]),
 ("Aseo del hogar",       ["bufalo","bolsa negra","betun","cera mansion","scotch brite","esponjilla","abrasivo",
                           "bon bril","vanish","desmanchador","desmanchado","blancox","limpia vidrios",
                           "quita pelusa","lozacrem","easy off"]),  ("Bebidas sin alcohol",  ["refresco","squash","instantaneo","sabiya","soy plus","bebida de soya","yoghurt para beber"]),
 ("Lácteos y huevos",     ["yoghurt","yogurth","deja mu","nata","lactosa","quesito","alpinette"]),
 ("Helados y postres",    ["pudin","pudines","gelhada","gel hada","gel'hada","tarta","cheesecake"]),
 ("Snacks y confitería",  ["tostacos","inka chips","chips","masmelos","millows","paraguitas","lechecitas",
                           "cocaditas","coffee delight","bom bon","anillos de cebolla","anillos cebolla"]),
 ("Panadería y galletas", ["marinela","gansito","lonchimix","submarino","divertimix","max combi","palmeritas",
                           "panelin","baguette","mi lonchera"]),
 ("Aseo del hogar",       ["limpido","niagara","papel aluminio","zeux","pano en rollo","mopa","balde",
                           "quitamanchas","beckman","panuelos","kleenex","caiman","pelusa"]),
 ("Cuidado personal",     ["floss","cera peinar","ego look","stayfree","tena","toallas higienicas"]),
 ("Licores y cervezas",   ["olmeca","reposado","pina colada"]),
 ("Carnes y embutidos",   ["prosciutto","albondiga"]),
 ("Abarrotes",            ["pepinillo","datiles","cerezas","marraschino","cebollitas","frasco premium","cocoa"]),  ("Snacks y confitería",  ["tajitas","tajaditas","piquitos","pretzels","trocitos de banano","chocmelos"]),
 ("Panadería y galletas", ["mantecada","concha","milhoja"]),
 ("Abarrotes",            ["cuadripanela","florentina","vegetales","duraznos","coctel","granola","syrup",
                           "sorrentinos","spaghettoni","esencia","aji","mccain"]),
 ("Aseo del hogar",       ["limpiapisos","toallas de cocina","pano scott","duramax"]),
 ("Belleza y maquillaje", ["lima manicure","perfilador","scunci","caiman","caucho"]),
 ("Cuidado personal",     ["maquina","body spray","fragancia","axe","pastillero"]),
 ("Licores y cervezas",   ["havana","anejo"]),
 ("Lácteos y huevos",     ["parmalat","surtilonchera","yagurt"]),
 ("Carnes y embutidos",   ["quibbe","farah","cabano"]),  ("Aseo del hogar", ["guarda polvo","guardapolvo","microfibra"]),
]

PRINCIPAL = [
 ("Cuidado personal", ["crema dental","pasta dental","gel dental","enjuague bucal","cepillo dental","colgate",
    "oral-b","oral b","shampoo","champu","acondicionador","tratamiento","capilar","cabello","gomina",
    "desodorante","antitranspirante","toalla higienica","toallita","panal","panitos","intimo","tampon",
    "afeitar","protex","dove","rexona","jabon de tocador","jabon de manos","gel antibacterial",
    "gel de ducha","gel para cabello","gel fijador","gel corporal","protector solar","talco","ducha",
    "crema para peinar","laca","protector diario","protectores diarios","nosotras","stayfree",
    "cortaunas","pañal"]),
 ("Belleza y maquillaje", ["esmalte","quitaesmalte","labial","vogue","masglo","sombra","delineador","pestanina",
    "pestanas","rubor","maquillaje","desmaquillador","tinte","coloracion","limpiador facial","agua micelar",
    "crema facial","crema corporal","crema de manos","crema para manos","locion","perfume","colonia",
    "polvo compacto","uñas","oxigenta","palette","oxidante","decolorante"]),
 ("Aseo del hogar", ["detergente","desinfectante","limpiador","ambientador","aromatizante","insecticida",
    "lavaloza","lavavajilla","lavaplatos","quita grasa","quitagrasa","blanqueador","suavizante","cloro",
    "clorox","glade","fantastik","fantastic","jabon","azulk","esponja","trapero","limpiavidrios",
    "multiusos","bicloro","preplanchado","almidon","repuesto","doypack","papel higienico","servilleta",
    "toalla de cocina","limpion","guante","escoba","ariel","fab","jbo home","bolsa de basura",
    "bolsa para basura","task","protecto","basura","panos humedos","mil usos","tanque","desengrasante",
    "maxima","rodenticida","raticida"]),
 ("Licores y cervezas", ["vino","cerveza","aguardiente","ron","whisky","vodka","tequila","ginebra","cabernet",
    "sauvignon","merlot","malbec","chardonnay","brandy","licor","sidra","champana","pinot","carmenere","tinto"]),
 ("Abarrotes", ["leche de coco","caldo","sopa","consome","atun","sardina"]),
 ("Panadería y galletas", ["pan","galleta","tostada","tostaos","bimbo","torta","arepa","bizcocho","ponque",
    "mogolla","croissant","pandebono","bunuelo","tortilla","wafer","cracker","saltin","dux"]),
 ("Helados y postres", ["helado","paleta","gelatina","flan","postre","natilla","tarro","merengue",
    "cheese cake","brownie"]),
 ("Snacks y confitería", ["papas fritas","pasabocas","snack","mani","chicle","caramelo","gomita","chocolatina",
    "bombon","doritos","cheetos","tostitos","goma","trululu","tic tac","vidal","criollitos","quipitos",
    "ramito","achira","barquillo","pistacho","pastilla"]),
 ("Lácteos y huevos", ["lacteo","leche condensada","crema de leche","leche","queso","yogurt","yogur","kumis",
    "mantequilla","huevo","alpina","colanta","alqueria","cuajada","margarina","suero","yogo"]),
 ("Carnes y embutidos", ["pollo","carne","cerdo","res","jamon","salchicha","salchichon","chorizo","tocineta",
    "mortadela","pescado","empanada","zenu","costilla","hamburguesa","nugget","pernil","butifarra",
    "morcilla","chicharron","pechuga","salami","peperoni","madurado","lomo","serrano"]),
 ("Bebidas sin alcohol", ["bebida","gaseosa","jugo","agua","refresco","energizante","isotonico","postobon",
    "coca cola","pepsi","sprite","malta","limonada","te liquido","nectar","tonica","soda"]),
 ("Mascotas", ["perro","gato","mascota","dog chow","cat chow","pedigree","whiskas"]),
 ("Abarrotes", ["arroz","pasta","azucar","sal","aceite","salsa","mermelada","panela","harina","avena","cereal",
    "lenteja","frijol","garbanzo","vinagre","especia","canela","comino","sazonador","condimento","cocoa",
    "miel","maizena","maiz","conserva","enlatado","pure","levadura","bicarbonato","aceituna","albahaca",
    "oregano","pimienta","laurel","tomillo","ajo","mayonesa","mostaza","ketchup","mezcla","coco",
    "vinagreta","aderezo","arveja","champinon","choclo","spaghetti","espagueti","fusilli","rigatoni",
    "penne","fideo","macarron","tallarin","lasana","jengibre","granaroma","chefrito","sofrito"]),
 ("Snacks y confitería", ["chocolate","dulce","barra","frutos secos","nueces","almendra"]),
]

RESPALDO = [    # palabras ambiguas: solo si nada más coincide
 ("Cuidado personal", ["crema","gel","cepillo"]),
 ("Aseo del hogar",   ["aerosol","antibacterial","lavanda"]),
 ("Abarrotes",        ["polvo"]),
]

FRUTAS = [("Frutas y verduras", ["platano","banano","tomate","vegetales","zanahoria","cebolla","lechuga","aguacate"])]   # último recurso

REGLAS = PRIORIDAD + PRINCIPAL + RESPALDO + FRUTAS
REGLAS_RX = [(cat, re.compile(r"\b(?:" + "|".join(re.escape(normalizar(p)) for p in pats) + r")s?\b"))
             for cat, pats in REGLAS]

def clasificar(nombre):
    n = normalizar(nombre)
    for cat, rx in REGLAS_RX:
        if rx.search(n):
            return cat
    return "Sin clasificar"

# ── 2. Clasificación (la tabla producto no se modifica todavía) ──────────────────────
df["ingreso"] = df["precio_venta"] * df["cantidad"]
clasif = producto[["no_parte", "producto"]].copy()
clasif["categoria"] = clasif["producto"].apply(clasificar)
clasif["ingreso"] = clasif["no_parte"].map(df.groupby("no_parte")["ingreso"].sum())

# ── 3. Resultados ────────────────────────────────────────────────────────────────────
ok = clasif["categoria"] != "Sin clasificar"
print(f"Productos clasificados: {ok.mean():.1%}")
print(f"Ingresos cubiertos:     {clasif.loc[ok,'ingreso'].sum() / clasif['ingreso'].sum():.1%}\n")

resumen = (clasif.groupby("categoria")
           .agg(productos=("no_parte", "count"), ingreso=("ingreso", "sum"))
           .assign(pct_ingreso=lambda t: (t["ingreso"] / t["ingreso"].sum() * 100).round(1))
           .sort_values("ingreso", ascending=False))
print(resumen, "\n")

print("── 40 pendientes con más ingreso ──")
print(clasif[~ok].sort_values("ingreso", ascending=False)
      .head(40)[["producto", "ingreso"]].to_string(index=False), "\n")

print("── Muestra de control (3 por categoría) ──")
muestra = clasif[ok].sample(frac=1, random_state=7).groupby("categoria").head(3)
print(muestra[["categoria", "producto"]].sort_values("categoria").to_string(index=False), "\n")

print("── Revisión de Frutas y verduras ──")
print(clasif[clasif["categoria"] == "Frutas y verduras"]["producto"].head(15).to_string(index=False))

Productos clasificados: 93.1%
Ingresos cubiertos:     99.4%

                      productos   ingreso  pct_ingreso
categoria                                             
Abarrotes                  1049  71381672        17.70
Belleza y maquillaje        543  57157484        14.20
Bebidas sin alcohol         560  56178490        13.90
Aseo del hogar             1126  41658225        10.30
Panadería y galletas        593  38437613         9.50
Snacks y confitería         666  33256318         8.20
Lácteos y huevos            743  30732113         7.60
Cuidado personal            687  23759698         5.90
Licores y cervezas          534  17148808         4.20
Carnes y embutidos          242  16154756         4.00
Helados y postres           215  13794130         3.40
Sin clasificar              518   2323485         0.60
Frutas y verduras            12   1538231         0.40 

── 40 pendientes con más ingreso ──
                                                     producto  ingreso
     

Muestra de control para re clasificar filas erroneas

In [ ]:
control = clasif[clasif["categoria"] != "Sin clasificar"].sample(100, random_state=11)[["no_parte","producto","categoria"]]
control["correcta"] = ""      # el grupo marca sí / no
control.to_csv("muestra_control_categorias.csv", index=False, encoding="utf-8-sig")

Descarga de tabla producto

In [ ]:
# Tabla producto: id_producto (= no_parte), categoria, producto, unidad_negocio
tabla_producto = producto[["no_parte", "producto", "unidad_negocio"]].copy()
tabla_producto["categoria"] = (
    tabla_producto["no_parte"].map(dict(zip(clasif["no_parte"], clasif["categoria"])))
    .replace("Sin clasificar", "Otros / sin clasificar")
)

# El número de parte pasa a ser el id del producto
tabla_producto = tabla_producto.rename(columns={"no_parte": "id_producto"})
tabla_producto = tabla_producto[["id_producto", "categoria", "producto", "unidad_negocio"]]
tabla_producto = tabla_producto.sort_values("id_producto").reset_index(drop=True)

# Validaciones
assert tabla_producto["id_producto"].is_unique
assert tabla_producto["id_producto"].notna().all()
assert tabla_producto["categoria"].notna().all()
assert len(tabla_producto) == df["no_parte"].nunique()

tabla_producto.to_csv("producto.csv", index=False, encoding="utf-8-sig")
print(tabla_producto.shape)
display(tabla_producto.head())

# Descarga en Colab
from google.colab import files
files.download("producto.csv")

(7488, 4)


,id_producto,categoria,producto,unidad_negocio
0,P100055,Abarrotes,Pasta Capelli D´Angelo Conzazoni,Supermercado
1,P100083,Carnes y embutidos,Salchicha Rica,Supermercado
2,P100512,Otros / sin clasificar,Maizitos,Supermercado
3,P100675,Abarrotes,Leche de coco Badia,Supermercado
4,P100765,Aseo del hogar,Guarda polvo microfibra Ubl,Supermercado


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

**Creación Tabla Cliente**

In [4]:
import numpy as np, pandas as pd

rng = np.random.default_rng(42)

nombres = ["Juan","Carlos","Luis","José","Miguel","Pedro","Jorge","Andrés","Diego","Felipe","Ricardo","Fernando","Pablo","Sergio","Manuel","Rodrigo","Eduardo","Héctor","Raúl","Daniel",
           "María","Ana","Carmen","Rosa","Lucía","Patricia","Laura","Sofía","Daniela","Valentina","Camila","Paula","Andrea","Claudia","Gabriela","Verónica","Natalia","Marcela","Fernanda","Isabel"]
apellidos = ["García","Rodríguez","González","Fernández","López","Martínez","Sánchez","Pérez","Gómez","Díaz","Torres","Ramírez","Flores","Vargas","Castillo","Rojas","Silva","Morales","Herrera","Mendoza",
             "Reyes","Cruz","Ortiz","Navarro","Medina","Romero","Soto","Contreras","Vega","Molina","Paredes","Campos","Núñez","Aguilar","Ríos","Salazar","Fuentes","Cortés","Espinoza","Guzmán"]

N = 2000
nombres_usados, telefonos_usados, registros = set(), set(), []

while len(registros) < N:
    nombre = f"{rng.choice(nombres)} {rng.choice(apellidos)} {rng.choice(apellidos)}"
    telefono = f"+56 9 {rng.integers(1000,10000)} {rng.integers(1000,10000)}"
    if nombre in nombres_usados or telefono in telefonos_usados:
        continue
    nombres_usados.add(nombre)
    telefonos_usados.add(telefono)
    registros.append((nombre, telefono))

cliente = pd.DataFrame(registros, columns=["nombre", "contacto"])
cliente.insert(0, "id_cliente", range(1, N + 1))

# Validaciones
assert cliente["id_cliente"].is_unique
assert cliente["nombre"].is_unique
assert cliente["contacto"].is_unique
assert cliente["contacto"].str.len().max() <= 100

cliente.to_csv("cliente.csv", index=False, encoding="utf-8-sig")
print(cliente.shape)
display(cliente.head())

# Descarga en Colab
from google.colab import files
files.download("cliente.csv")

(2000, 3)


,id_cliente,nombre,contacto
0,1,José Paredes Soto,+56 9 4949 4897
1,2,Gabriela Fernández Contreras,+56 9 2813 1847
2,3,Ana Guzmán Molina,+56 9 7850 7457
3,4,Paula Reyes Martínez,+56 9 8557 5053
4,5,María Castillo Pérez,+56 9 9340 8034


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

**Creación Tabla Venta**

La tabla `venta` se construyó a partir de `ventas_originales_fuente.csv` (103.489 líneas de venta), con una fila por factura.
El prefijo de la columna `factura` (por ejemplo `INV7778844`) se repite en fechas distintas, por lo que no identifica una factura única.
Por eso el `id_factura` se definió como prefijo + fecha + tipo de compra (F = Física, L = En Línea), por ejemplo `INV7811414-20100105-F`.
Con esa definición cada factura tiene una sola fecha y un solo tipo de compra, y resultan 100.067 facturas.
La `fecha` se convirtió a formato fecha y el `tipo_compra` se tomó directamente del archivo original.
Como la fuente no trae clientes, a cada factura se le asignó un `id_cliente` al azar entre los 2.000 clientes ficticios de la tabla `cliente`.
El campo `por_cobrar` se generó de forma aleatoria, marcando aproximadamente el 15 % de las facturas como pendientes de cobro.
Se validó que `id_factura` fuera único, que todos los `id_cliente` existieran en `cliente` y que `fecha_cobro` fuera posterior a `fecha`.
El resultado se exportó como `venta.csv` y la columna `id_factura` se conserva en el DataFrame de origen para armar `Detalle_Venta`.

In [5]:
import numpy as np, pandas as pd

rng = np.random.default_rng(42)

# 1) Leer ventas originales
df = pd.read_csv("ventas_originales_fuente.csv")
df["fecha"] = pd.to_datetime(df["fecha"], utc=True).dt.tz_localize(None).dt.normalize()

# 2) id_factura = prefijo + fecha + tipo (F = Física, L = En Línea)
base = df["factura"].str.split("-").str[0]
tipo = df["tipo_compra"].map({"Física": "F", "En Línea": "L"})
df["id_factura"] = base + "-" + df["fecha"].dt.strftime("%Y%m%d") + "-" + tipo

# 3) Una fila por factura
venta = (
    df.groupby("id_factura", as_index=False)
      .agg(fecha=("fecha", "first"), tipo_compra=("tipo_compra", "first"))
      .sort_values(["fecha", "id_factura"])
      .reset_index(drop=True)
)

# 4) Campos ficticios: cliente y por cobrar (15%)
n = len(venta)
venta["id_cliente"] = rng.choice(cliente["id_cliente"].to_numpy(), size=n)
venta["por_cobrar"] = rng.random(n) < 0.15

venta = venta[["id_factura", "id_cliente", "fecha", "tipo_compra", "por_cobrar"]]

# Validaciones
assert venta["id_factura"].is_unique
assert venta["id_factura"].str.len().max() <= 30
assert venta["id_cliente"].isin(cliente["id_cliente"]).all()

venta.to_csv("venta.csv", index=False, encoding="utf-8-sig")
print(venta.shape, f"por cobrar: {venta['por_cobrar'].mean():.1%}")

# Descarga en Colab
from google.colab import files
files.download("venta.csv")

(100067, 5) por cobrar: 14.9%


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>


**Creación Tabla Detalle Venta**

La tabla `detalle_venta` se construyó a partir de `ventas_originales_fuente.csv`, con una fila por cada línea de venta (103.489 filas).
Cada línea se asoció a su factura mediante el `id_factura` definido en la tabla `venta` (prefijo + fecha + tipo de compra).
El número de parte (`no_parte`) se renombró como `id_producto`, ya que es el identificador de la tabla `producto`.
Se conservaron `cantidad`, `precio_venta` y `costo_venta` tal como vienen en el archivo original, sin modificarlos.
Las líneas se ordenaron por fecha, factura y producto, y sobre ese orden se generó `id_detalle` como un número autoincremental desde 1.
Se fijaron los tipos de dato: `id_factura` e `id_producto` como texto, y `id_detalle` y `cantidad` como enteros de 32 bits, y `precio_venta` y `costo_venta` como enteros de 64 bits.
Se validó que `id_detalle` fuera único y que el total de filas coincidiera con el archivo original.
También se verificó que todos los `id_factura` existieran en `venta` y todos los `id_producto` en `producto`, para asegurar las claves foráneas.
Por último, se comprobó que no hubiera nulos y que cantidad, precio y costo fueran siempre positivos.
El resultado se exportó como `detalle_venta.csv`.

In [8]:
# Detalle_Venta: una fila por línea de venta original
tabla_producto = pd.read_csv("producto.csv")
detalle_venta = (
    df[["id_factura", "no_parte", "cantidad", "precio_venta", "costo_venta", "fecha", "source_row_id"]]
    .rename(columns={"no_parte": "id_producto"})
    .sort_values(["fecha", "id_factura", "id_producto", "source_row_id"])
    .reset_index(drop=True)
)

# id_detalle autoincremental
detalle_venta.insert(0, "id_detalle", range(1, len(detalle_venta) + 1))

detalle_venta = detalle_venta[
    ["id_detalle", "id_factura", "id_producto", "cantidad", "precio_venta", "costo_venta"]
]

# Tipos de dato
detalle_venta = detalle_venta.astype({
    "id_detalle": "int32",
    "id_factura": "string",
    "id_producto": "string",
    "cantidad": "int32",
    "precio_venta": "int64",
    "costo_venta": "int64",
})

# Validaciones
assert detalle_venta["id_detalle"].is_unique
assert len(detalle_venta) == len(df)
assert detalle_venta["id_factura"].isin(venta["id_factura"]).all()             # FK a venta
assert detalle_venta["id_producto"].isin(tabla_producto["id_producto"]).all()  # FK a producto
assert detalle_venta[["cantidad", "precio_venta", "costo_venta"]].notna().all().all()
assert (detalle_venta[["cantidad", "precio_venta", "costo_venta"]] > 0).all().all()
assert detalle_venta["id_factura"].str.len().max() <= 30

detalle_venta.to_csv("detalle_venta.csv", index=False, encoding="utf-8-sig")
print(detalle_venta.shape)
print(detalle_venta.dtypes)
display(detalle_venta.head())

# Descarga en Colab
from google.colab import files
files.download("detalle_venta.csv")

(103489, 6)
id_detalle               int32
id_factura      string[python]
id_producto     string[python]
cantidad                 int32
precio_venta             int64
costo_venta              int64
dtype: object


,id_detalle,id_factura,id_producto,cantidad,precio_venta,costo_venta
0,1,INV7811414-20100105-F,P126966,1,777,665
1,2,INV7810919-20100107-F,P874751,1,525,452
2,3,INV7811412-20100107-L,P489329,1,3288,2827
3,4,INV7811577-20100108-L,P626794,2,803,690
4,5,INV7810336-20100110-F,P996462,2,1157,1106


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

**Creación Tabla Cuentas por Cobrar**

La tabla cuenta_por_cobrar se construyó a partir de las tablas venta y detalle_venta, con una fila por cada factura marcada como por_cobrar. Esta versión corresponde al MVP, cuyo objetivo es únicamente advertir los vencimientos de cobro; los campos estado y fecha_pago se dejan para una mejora posterior. Resultaron 14.928 cuentas, que corresponden a cerca del 15 % de las 100.067 facturas. El monto de cada cuenta se calculó como la suma de cantidad × precio_venta de todas las líneas de la factura en detalle_venta. Los montos van desde 34 hasta 880.999, valores que caben holgadamente en el límite de int(8) definido en el diagrama. El id_cliente se tomó de la tabla venta, de modo que cada cuenta queda asociada al mismo cliente de su factura. La fecha_vencimiento es un dato ficticio que se calcula en esta tabla, sumando entre 15 y 60 días a la fecha de la factura; esta fecha reemplaza a la antigua fecha_cobro, que se eliminó de venta para evitar duplicidad. El id_cuenta se generó como número autoincremental desde 1, ordenando por fecha de vencimiento y factura. Como hay 14.928 cuentas, id_cuenta supera el límite de int(4) (9.999), por lo que en el diagrama debe pasar a int(5). Se fijaron los tipos de dato y se validó que id_factura fuera único (relación 1 a 0..1 con venta) y que las claves foráneas existieran en venta y cliente. También se verificó que los montos y las fechas de vencimiento no tuvieran nulos y que los montos fueran positivos, y el resultado se exportó como cuenta_por_cobrar.csv.

In [13]:
# Cuenta_por_cobrar: una fila por cada factura marcada como por_cobrar
# MVP: solo permite alertar vencimientos. El estado (pagada/pendiente) se agregará en una mejora.

# 1) Monto de cada factura = suma de cantidad * precio_venta
totales = (
    detalle_venta.assign(subtotal=detalle_venta["cantidad"] * detalle_venta["precio_venta"])
    .groupby("id_factura", as_index=False)
    .agg(monto=("subtotal", "sum"))
)

# 2) Solo facturas por cobrar; id_cliente y fecha vienen de venta
rng_cc = np.random.default_rng(43)
cuenta_por_cobrar = (
    venta.loc[venta["por_cobrar"], ["id_factura", "id_cliente", "fecha"]]
    .merge(totales, on="id_factura", how="left")
    .reset_index(drop=True)
)

# Vencimiento ficticio: 15 a 60 días después de la venta
dias = rng_cc.integers(15, 61, size=len(cuenta_por_cobrar))
cuenta_por_cobrar["fecha_vencimiento"] = cuenta_por_cobrar["fecha"] + pd.to_timedelta(dias, unit="D")
cuenta_por_cobrar = cuenta_por_cobrar.sort_values(["fecha_vencimiento", "id_factura"]).reset_index(drop=True)

# 3) id_cuenta autoincremental
cuenta_por_cobrar.insert(0, "id_cuenta", range(1, len(cuenta_por_cobrar) + 1))

cuenta_por_cobrar = cuenta_por_cobrar[
    ["id_cuenta", "id_factura", "id_cliente", "monto", "fecha_vencimiento"]
]

# Tipos de dato
cuenta_por_cobrar = cuenta_por_cobrar.astype({
    "id_cuenta": "int32",
    "id_factura": "string",
    "id_cliente": "int32",
    "monto": "int64",
})

# Validaciones
assert cuenta_por_cobrar["id_cuenta"].is_unique
assert cuenta_por_cobrar["id_factura"].is_unique                               # relación 1 a 0..1 con venta
assert len(cuenta_por_cobrar) == venta["por_cobrar"].sum()
assert cuenta_por_cobrar["id_factura"].isin(venta["id_factura"]).all()         # FK a venta
assert cuenta_por_cobrar["id_cliente"].isin(cliente["id_cliente"]).all()       # FK a cliente
assert cuenta_por_cobrar["monto"].notna().all() and (cuenta_por_cobrar["monto"] > 0).all()
assert cuenta_por_cobrar["fecha_vencimiento"].notna().all()
assert cuenta_por_cobrar["monto"].max() < 10**8                                # cabe en int(8)
assert cuenta_por_cobrar["id_cuenta"].max() < 10**5                            # cabe en int(5)

cuenta_por_cobrar.to_csv("cuenta_por_cobrar.csv", index=False, encoding="utf-8-sig")
print(cuenta_por_cobrar.shape)
print(cuenta_por_cobrar.dtypes)
display(cuenta_por_cobrar.head())

# Descarga en Colab
from google.colab import files
files.download("cuenta_por_cobrar.csv")

(14928, 5)
id_cuenta                     int32
id_factura           string[python]
id_cliente                    int32
monto                         int64
fecha_vencimiento    datetime64[ns]
dtype: object


,id_cuenta,id_factura,id_cliente,monto,fecha_vencimiento
0,1,INV7811601-20100127-L,887,25893,2010-02-11
1,2,INV7809940-20100127-F,1646,2876,2010-02-13
2,3,INV7811412-20100107-L,1310,3288,2010-02-14
3,4,INV7811595-20100128-F,1776,730,2010-02-24
4,5,INV7811638-20100126-F,805,6150,2010-02-28


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

**Creación Tabla Gastos Insumo**

La tabla `gastos_insumo` es ficticia, porque el archivo de ventas no trae información de gastos, y solo depende de la tabla `producto`.
Se generaron 20.000 registros, cantidad que supera el límite de `int(4)` (9.999) definido para `id_gasto`, por lo que en el diagrama debe pasar a `int(5)`.
Los productos se eligieron al azar entre los que aparecen en `detalle_venta`, y resultaron 6.973 productos distintos con al menos un gasto.
Las fechas se sortearon dentro del rango de las ventas, entre el 5 de enero de 2010 y el 18 de noviembre de 2022.
El `Monto` se calculó a partir del costo de venta promedio de cada producto, multiplicado por un factor aleatorio entre 0,8 y 1,2 (±20 %) y redondeado a entero.
Los montos resultantes van desde 21 hasta 17.860, valores que caben en el límite de `int(10)` definido en el diagrama.
Los registros se ordenaron por fecha y producto, y sobre ese orden se generó `id_gasto` como número autoincremental desde 1.
Se fijaron los tipos de dato: `id_producto` como texto y `id_gasto` como entero de 32 bits, y `Monto` como entero de 64 bits.
Se validó que `id_gasto` fuera único, que todos los `id_producto` existieran en `producto` y que las fechas estuvieran dentro del rango de ventas.
También se comprobó que los montos fueran positivos y no nulos, y el resultado se exportó como `gastos_insumo.csv`.

In [ ]:
import numpy as np, pandas as pd

rng = np.random.default_rng(42)

N = 20000  # cantidad de gastos

# 1) Costo promedio por producto (base del monto)
costo_prom = detalle_venta.groupby("id_producto")["costo_venta"].mean()

# 2) Productos al azar, solo entre los que se vendieron
ids = rng.choice(costo_prom.index.to_numpy(), size=N)

# 3) Fechas al azar dentro del rango de las ventas
f_min, f_max = venta["fecha"].min(), venta["fecha"].max()
dias = rng.integers(0, (f_max - f_min).days + 1, size=N)

# 4) Monto cercano al costo de venta, con variación de ±20 %
monto = (costo_prom.loc[ids].to_numpy() * rng.uniform(0.8, 1.2, size=N)).round().astype("int64")

gastos_insumo = pd.DataFrame({
    "id_producto": ids,
    "Fecha": f_min + pd.to_timedelta(dias, unit="D"),
    "Monto": np.maximum(monto, 1),
})

# 5) Orden e id_gasto autoincremental
gastos_insumo = gastos_insumo.sort_values(["Fecha", "id_producto"]).reset_index(drop=True)
gastos_insumo.insert(0, "id_gasto", range(1, N + 1))

# Tipos de dato
gastos_insumo = gastos_insumo.astype({
    "id_gasto": "int32",
    "id_producto": "string",
    "Monto": "int64",
})

# Validaciones
assert gastos_insumo["id_gasto"].is_unique
assert gastos_insumo["id_producto"].isin(tabla_producto["id_producto"]).all()   # FK a producto
assert gastos_insumo["Fecha"].between(f_min, f_max).all()
assert gastos_insumo["Monto"].notna().all() and (gastos_insumo["Monto"] > 0).all()
assert gastos_insumo["Monto"].max() < 10**10                                    # cabe en int(10)

gastos_insumo.to_csv("gastos_insumo.csv", index=False, encoding="utf-8-sig")
print(gastos_insumo.shape)
print(gastos_insumo.dtypes)
display(gastos_insumo.head())

# Descarga en Colab
from google.colab import files
files.download("gastos_insumo.csv")

(20000, 4)
id_gasto                int32
id_producto    string[python]
Fecha          datetime64[ns]
Monto                   int64
dtype: object


,id_gasto,id_producto,Fecha,Monto
0,1,P161900,2010-01-05,88
1,2,P209965,2010-01-05,149
2,3,P840906,2010-01-05,176
3,4,P850047,2010-01-05,3170
4,5,P859698,2010-01-05,31


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>